# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [21]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [22]:
q1=q('''SELECT t.track_id, a.name, a.country
FROM tracks t
LEFT JOIN artists a on t.artist_id = a.artist_id
''')

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [23]:
q('''SELECT t.genre, AVG(t.seconds)/60.0 AS avg_minutes
FROM tracks t
GROUP BY t.genre
ORDER by avg_minutes DESC
LIMIT 1
''')

,genre,avg_minutes
0,Electronic,4.791667


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [24]:
q3=q('''SELECT p.user, COUNT(p.play_id) AS plays, COUNT(DISTINCT p.track_id) AS distinct_tracks
FROM plays p
GROUP BY p.user
''')

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [25]:
q4=q('''SELECT t.title text
FROM tracks t
LEFT JOIN plays p on p.track_id = t.track_id
WHERE p.play_id is NULL

''')

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [26]:
q('''SELECT a.name, ROUND(SUM(t.seconds)/ 60.0,1) AS total_minutes
FROM tracks t
JOIN plays p ON p.track_id = t.track_id
JOIN artists a ON t.artist_id=a.artist_id
GROUP BY a.name
ORDER by total_minutes DESC
''')

,name,total_minutes
0,Kestrel,19.6
1,Nova Waves,14.1
2,The Blue Ridge,6.4
3,Marisol,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [27]:
q('''SELECT t.title, t.track_id
FROM tracks t
WHERE genre is NULL

''')

,title,track_id
0,Untitled Demo,18


In [28]:
q('''SELECT t.title, t.track_id
FROM tracks t
WHERE genre != 'Pop'

''')

,title,track_id
0,Foothills,12
1,Aurora,13
2,Nightfall,14
3,Sol,15
4,Coastline,16
5,Ridgeline,17


In [29]:
#Running WHERE genre != 'Pop' results in the exlusion of the Untited Demo track. Even though it's genre isn't pop, it's genre is Null, which means SQL excludes it.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [30]:
q('''SELECT played_on, COUNT(play_id)AS total_plays, COUNT (DISTINCT user) as distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC

''')

,played_on,total_plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [31]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I had the most trouble with Query 5. I knew that I would eventually need to group by name, but it took me a while to realize I also needed to select a name in the first row for the output to have an artist name column.